# Analisis potensi site Telkomsel
Notebook KP: kelas payload, digital, games, video dan karakteristik setiap kelas.

Unit analisis adalah site-bulan. Kelas menunjukkan posisi relatif historis, bukan prediksi pelanggan individual. Jalankan sel berurutan. Seluruh kode pengolahan tersedia di notebook, tanpa ketergantungan pada skrip lokal.

Untuk Colab: tempatkan CSV pada Google Drive, mount Drive dengan `from google.colab import drive; drive.mount("/content/drive")`, kemudian ubah INPUT pada sel konfigurasi. File sekitar 1,8 GiB dibaca bertahap; siapkan RAM dan ruang penyimpanan yang memadai. Output disimpan pada direktori kerja notebook.

In [ ]:
from google.colab import drive
from pathlib import Path
import os

drive.mount('/content/drive')
os.chdir('/content/drive/MyDrive/insight-kp')


In [ ]:
INPUT = Path('/content/drive/MyDrive/insight-kp/site_potential_daily 2.csv')


In [ ]:
from pathlib import Path

INPUT = Path('/content/drive/MyDrive/insight-kp/site_potential_daily 2.csv')
assert INPUT.is_file(), "CSV belum ditemukan. Periksa lokasi file."
print("CSV ditemukan, siap diproses.")


## 1. Audit dan agregasi CSV asli
Validasi data dan agregasi rata-rata per hari site tercatat. Hari tanpa data tidak dianggap nol. Label payload lama hanya dipertahankan sebagai pembanding; empat kelas utama dibuat pada tahap berikutnya.

In [ ]:
import sys
sys.argv = ["olah_data", "--input", str(INPUT), "--output", "agregat"]
"""Rebuild local Tableau inputs from daily site data; never modifies the source."""
from pathlib import Path
import argparse
import hashlib
import json
import numpy as np
import pandas as pd

METRICS = ['subs', 'sdn', 'rgb', 'du', 'payload_user', 'digital_user', 'games_user', 'video_user']

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--input', type=Path, required=True)
    parser.add_argument('--output', type=Path, default=Path('agregat'))
    args = parser.parse_args()
    args.output.mkdir(parents=True, exist_ok=True)
    monthly_parts, daily_parts, keys = [], [], []
    row_count = 0
    total = pd.Series(0.0, index=METRICS)
    for chunk in pd.read_csv(args.input, chunksize=250_000, dtype={'site_id': 'string'}):
        if list(chunk.columns) != ['event_date', 'site_id'] + METRICS:
            raise ValueError('Unexpected source columns')
        if chunk.isna().any().any():
            raise ValueError('Missing values require investigation; no automatic imputation applied')
        chunk['event_date'] = pd.to_datetime(chunk['event_date'], format='%Y-%m-%d', errors='raise')
        if not np.isfinite(chunk[METRICS].to_numpy()).all() or chunk[METRICS].lt(0).any().any():
            raise ValueError('Invalid metric values')
        keys.append(pd.util.hash_pandas_object(chunk[['event_date','site_id']],index=False).to_numpy())
        chunk['month'] = chunk.event_date.dt.to_period('M').dt.to_timestamp()
        chunk['days_observed'] = 1
        chunk['games_video_review_days'] = (chunk.event_date >= '2024-06-28').astype(int)
        chunk['payload_gt_du_days'] = (chunk.payload_user > chunk.du + 1e-8).astype(int)
        chunk['video_gt_du_days'] = (chunk.video_user > chunk.du + 1e-8).astype(int)
        measures = METRICS + ['days_observed','games_video_review_days','payload_gt_du_days','video_gt_du_days']
        monthly_parts.append(chunk.groupby(['site_id','month'], observed=True)[measures].sum())
        daily_parts.append(chunk.groupby('event_date')[measures].sum())
        total += chunk[METRICS].sum()
        row_count += len(chunk)
        print(f'Processed {row_count:,} rows', flush=True)
    hashes = np.concatenate(keys)
    if len(np.unique(hashes)) != row_count:
        raise ValueError('Repeated site-date hashes: inspect duplicate keys before aggregation')
    del keys, hashes
    monthly = pd.concat(monthly_parts).groupby(level=[0,1]).sum().reset_index()
    daily = pd.concat(daily_parts).groupby(level=0).sum()
    del monthly_parts, daily_parts
    assert not monthly.duplicated(['site_id','month']).any()
    assert monthly.days_observed.sum() == row_count
    assert np.allclose(monthly[METRICS].sum(), total, rtol=1e-12, atol=0.01)
    assert np.allclose(daily[METRICS].sum(), total, rtol=1e-12, atol=0.01)
    available = pd.Series(1, index=daily.index).groupby(daily.index.to_period('M').to_timestamp()).sum()
    monthly['days_available_in_file'] = monthly.month.map(available)
    monthly['days_in_month'] = monthly.month.dt.days_in_month
    monthly['site_coverage_available'] = monthly.days_observed / monthly.days_available_in_file
    monthly['calendar_coverage'] = monthly.days_observed / monthly.days_in_month
    monthly['coverage_status'] = np.where(monthly.days_observed.eq(monthly.days_available_in_file), 'Lengkap pada tanggal tersedia', 'Site tidak tercatat pada sebagian tanggal')
    for metric in METRICS:
        monthly[f'{metric}_daily_avg'] = monthly[metric] / monthly.days_observed
        monthly.rename(columns={metric: f'{metric}_observed_sum'}, inplace=True)
    # Fixed descriptive thresholds over the current historical file, not ML predictions.
    q1, q2 = monthly.payload_user_daily_avg.quantile([1/3, 2/3]).tolist()
    monthly['payload_class'] = pd.cut(monthly.payload_user_daily_avg, [-np.inf,q1,q2,np.inf], labels=['Rendah','Sedang','Tinggi'])
    old_q1, old_q2 = monthly.payload_user_observed_sum.quantile([1/3,2/3]).tolist()
    monthly['payload_class_legacy_sum'] = pd.cut(monthly.payload_user_observed_sum, [-np.inf,old_q1,old_q2,np.inf], labels=['Rendah','Sedang','Tinggi'])
    monthly['games_video_status'] = np.where(monthly.games_video_review_days.gt(0), 'Perlu tinjauan perubahan data sejak 2024-06-28', 'Sebelum perubahan 2024-06-28')
    monthly['payload_du_ratio'] = monthly.payload_user_observed_sum / monthly.du_observed_sum.replace(0,np.nan)
    monthly.sort_values(['month','site_id'], inplace=True)
    monthly.to_csv(args.output / 'master_site_bulanan.csv', index=False, date_format='%Y-%m-%d')
    # One row per calendar date, including absent dates with blank metrics.
    daily.rename(columns={'days_observed':'site_count'}, inplace=True)
    for metric in METRICS:
        daily[f'{metric}_site_avg'] = daily[metric] / daily.site_count
        daily.rename(columns={metric: f'{metric}_site_sum'}, inplace=True)
    daily = daily.reindex(pd.date_range(daily.index.min(), daily.index.max(), name='event_date'))
    daily['data_available'] = daily.site_count.notna()
    daily['games_video_status'] = np.where(daily.index >= pd.Timestamp('2024-06-28'), 'Perlu tinjauan', 'Sebelum perubahan')
    daily.to_csv(args.output / 'tren_harian.csv', date_format='%Y-%m-%d')
    quality = monthly.groupby('month').agg(site_count=('site_id','nunique'), site_days=('days_observed','sum'), days_available=('days_available_in_file','first'), days_in_month=('days_in_month','first'), fully_observed_sites=('site_coverage_available',lambda x: int(x.eq(1).sum())))
    quality['missing_calendar_days'] = quality.days_in_month - quality.days_available
    quality['calendar_coverage'] = quality.days_available / quality.days_in_month
    quality['month_note'] = 'Tanggal tidak tersedia belum tentu aktivitas nol'
    quality.to_csv(args.output / 'cakupan_bulanan.csv', date_format='%Y-%m-%d')
    # Count sites within each month; never sum distinct counts across months.
    classes = monthly.groupby(['month','payload_class'], observed=True).agg(site_count=('site_id','nunique'), mean_payload_daily=('payload_user_daily_avg','mean')).reset_index()
    classes.to_csv(args.output / 'distribusi_kelas_bulanan.csv',index=False,date_format='%Y-%m-%d')
    assert classes.site_count.sum() == len(monthly)
    check = pd.read_csv(args.output / 'master_site_bulanan.csv',usecols=['site_id','month','days_observed','payload_user_observed_sum','payload_class'])
    assert len(check) == len(monthly) and check.days_observed.sum() == row_count
    assert not check.isna().any().any()
    summary = {
        'source_file':str(args.input), 'source_bytes':args.input.stat().st_size,
        'source_rows':row_count, 'monthly_rows':len(monthly), 'unique_sites':int(monthly.site_id.nunique()),
        'first_date':str(daily.index.min().date()),'last_date':str(daily.index.max().date()),
        'dates_available':int(daily.data_available.sum()),'missing_dates_in_range':int((~daily.data_available).sum()),
        'payload_daily_avg_thresholds':{'low_max':q1,'medium_max':q2},
        'legacy_observed_sum_thresholds':{'low_max':old_q1,'medium_max':old_q2},
        'class_method':'Fixed global tertiles over site-month daily averages in this historical file; descriptive, not ML predictions',
        'class_changes_vs_legacy':int((monthly.payload_class != monthly.payload_class_legacy_sum).sum()),
        'validation':'Source and aggregates reconciled; site-month unique; source key hashes unique; exported master reread',
        'software':{'pandas':pd.__version__,'numpy':np.__version__},
        'files_sha256':{p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in sorted(args.output.glob('*.csv'))}
    }
    (args.output / 'hasil_validasi.json').write_text(json.dumps(summary,indent=2))
    print(json.dumps(summary,indent=2))

if __name__ == '__main__':
    main()


## 2. Empat klasifikasi dan karakteristik
Batas tertil tetap dihitung terpisah untuk setiap kategori. Profil meliputi jumlah observasi, median delapan metrik, rentang kuartil, cakupan hari, dan kontribusi jumlah teramati. Ambang deskriptif memakai seluruh periode; jangan digunakan sebagai evaluasi prediksi.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

ROOT = Path.cwd() if Path('agregat').exists() else Path('insight-kp')
OUT = ROOT / 'hasil'
OUT.mkdir(parents=True, exist_ok=True)
df = pd.read_csv(ROOT / 'agregat/master_site_bulanan.csv')
daily = pd.read_csv(ROOT / 'agregat/tren_harian.csv', parse_dates=['event_date'])
audit = json.loads((ROOT / 'agregat/hasil_validasi.json').read_text())
cats = ['payload', 'digital', 'games', 'video']
metrics = ['subs', 'sdn', 'rgb', 'du'] + [c+'_user' for c in cats]
labels = ['Rendah', 'Sedang', 'Tinggi']
thresholds, profiles, distributions, sensitivity = [], [], [], []
for c in cats:
    col = c+'_user_daily_avg'
    lo, hi = df[col].quantile([1/3, 2/3])
    df[c+'_kelas'] = pd.cut(df[col], [-np.inf,lo,hi,np.inf], labels=labels)
    thresholds.append({'kategori':c, 'rendah_maks':lo,'sedang_maks':hi})
    for label in labels:
        part = df[df[c+'_kelas']==label]
        row = {'kategori':c,'kelas':label,'site_bulan':len(part),'persen_site_bulan':100*len(part)/len(df),
               'median_hari_teramati':part.days_observed.median(),
               'persen_cakupan_site_lengkap':100*part.site_coverage_available.eq(1).mean(),
               'kontribusi_jumlah_teramati_pct':100*part[c+'_user_observed_sum'].sum()/df[c+'_user_observed_sum'].sum()}
        for m in metrics:
            row[m+'_median'] = part[m+'_daily_avg'].median()
        row['target_p25'] = part[col].quantile(.25)
        row['target_p75'] = part[col].quantile(.75)
        profiles.append(row)
    counts = df.groupby(['month',c+'_kelas'],observed=True).size()
    counts = counts.reindex(pd.MultiIndex.from_product([sorted(df.month.unique()),labels], names=['month',c+'_kelas']),fill_value=0)
    dist = counts.rename('site').reset_index().rename(columns={c+'_kelas':'kelas'})
    dist['kategori'] = c
    dist['persen'] = dist.site / dist.groupby('month').site.transform('sum')*100
    distributions.append(dist)
    # Sensitivity: calibrate thresholds on months entirely before the flagged change.
    clean = df[df.month < '2024-06-01']
    a,b = clean[col].quantile([1/3,2/3])
    alt = pd.cut(df[col],[-np.inf,a,b,np.inf],labels=labels)
    sensitivity.append({'kategori':c,'batas_rendah_pra_juni':a,'batas_sedang_pra_juni':b,
                        'persen_berubah_semua_bulan':100*(alt!=df[c+'_kelas']).mean()})
thresholds = pd.DataFrame(thresholds)
profiles = pd.DataFrame(profiles)
dist = pd.concat(distributions,ignore_index=True)
sens = pd.DataFrame(sensitivity)
assert all(df[c+'_kelas'].notna().all() for c in cats)
assert profiles.groupby('kategori').site_bulan.sum().eq(len(df)).all()
assert np.allclose(profiles.groupby('kategori').kontribusi_jumlah_teramati_pct.sum(),100)
assert not df.duplicated(['site_id','month']).any()
thresholds.to_csv(OUT/'batas_kelas.csv',index=False)
profiles.to_csv(OUT/'karakteristik_kelas.csv',index=False)
dist.to_csv(OUT/'distribusi_bulanan.csv',index=False)
sens.to_csv(OUT/'sensitivitas_batas.csv',index=False)
df[['site_id','month','days_observed','site_coverage_available','calendar_coverage']+
   [m+'_daily_avg' for m in metrics]+[c+'_kelas' for c in cats]].to_csv(OUT/'kelas_site_bulanan.csv',index=False)



## 3. Grafik dan laporan insight
Grafik menunjukkan perubahan distribusi dan tren harian. Uji sensitivitas membandingkan batas historis penuh dengan batas sebelum Juni 2024. Rekomendasi bisnis merupakan hipotesis yang perlu diuji.

In [ ]:
plt.rcParams.update({'font.size':10,'axes.spines.top':False,'axes.spines.right':False})
fig,axes = plt.subplots(2,2,figsize=(12,7),layout='constrained')
for c,ax in zip(cats,axes.flat):
    for label,color in zip(labels,['#94a3b8','#38bdf8','#0f766e']):
        s = dist[(dist.kategori==c)&(dist.kelas==label)]
        ax.plot(s.month.str[:7],s.persen,label=label,marker='o',color=color)
    ax.set(title=c.capitalize(),ylabel='% site pada bulan tersebut',ylim=(0,100))
    ax.tick_params(axis='x',rotation=45)
axes[0,0].legend(ncol=3,fontsize=8)
fig.suptitle('Distribusi kelas bulanan dengan ambang historis tetap')
fig.savefig(OUT/'distribusi_kelas.png',dpi=160)
plt.close(fig)

fig,axes = plt.subplots(2,2,figsize=(12,7),layout='constrained')
for c,ax in zip(cats,axes.flat):
    ax.plot(daily.event_date,daily[c+'_user_site_avg'],color='#0f766e')
    ax.axvline(pd.Timestamp('2024-06-28'),color='#dc2626',ls='--')
    ax.set(title=c.capitalize(),ylabel='Rata-rata lintas site tercatat')
    ax.tick_params(axis='x',rotation=30)
fig.suptitle('Tren harian; celah garis berarti tanggal tanpa data')
fig.savefig(OUT/'tren_harian.png',dpi=160)
plt.close(fig)

def num(x):
    return f'{x:,.2f}'.replace(',','X').replace('.',',').replace('X','.')
def table(frame):
    return frame.to_html(index=False,border=0,float_format=num)

parts = ['<h1>Insight potensi site Telkomsel</h1>',
    '<p>Analisis kerja praktik • Sumber: site_potential_daily 2.csv • Periode '+audit['first_date']+' sampai '+audit['last_date']+'</p>',
    '<h2>Ringkasan dan ruang lingkup</h2>',
    f'<p>Data memuat {audit["source_rows"]:,} baris site-hari, {audit["unique_sites"]:,} site, dan {len(df):,} observasi site-bulan. Tersedia {audit["dates_available"]} tanggal; {audit["missing_dates_in_range"]} tanggal di dalam rentang tidak tersedia. Satu site dapat berpindah kelas antarbulan.</p>',
    '<p>Empat kategori diklasifikasikan secara terpisah menjadi Rendah, Sedang, dan Tinggi. Potensi di sini berarti posisi relatif besaran metrik historis. Data tidak mengidentifikasi pengguna individual dan tidak mengukur pendapatan tambahan atau permintaan yang belum terpenuhi.</p>',
    '<h2>Metode kelas</h2><p>Nilai site-bulan adalah jumlah metrik dibagi jumlah hari site tercatat. Ambang menggunakan persentil 33⅓ dan 66⅔ dari seluruh site-bulan, dengan bobot sama per site-bulan. Rendah: nilai ≤ batas pertama; Sedang: batas pertama &lt; nilai ≤ batas kedua; Tinggi: nilai &gt; batas kedua. Ambang tetap lintas bulan. Besaran kelas sekitar sepertiga merupakan konsekuensi metode, bukan penemuan bisnis. Angka tabel dibulatkan; CSV menyimpan presisi penuh.</p>',
    table(thresholds),
    '<p>Satuan payload_user, digital_user, games_user, video_user serta definisi subs, sdn, rgb, du belum dikonfirmasi melalui kamus data. Nilai pecahan tidak diperlakukan sebagai jumlah orang unik. Kategori dapat tumpang tindih dan tidak boleh dijumlahkan menjadi total pelanggan. Ini segmentasi deskriptif; tidak ada pelatihan model atau klaim akurasi prediksi.</p>']
actions = {
 'payload':['Periksa ukuran basis site dan hambatan pemakaian sebelum merancang aktivasi.','Uji penawaran paket data pada site yang menunjukkan kenaikan konsisten.','Prioritaskan pemeriksaan kapasitas dan kualitas layanan; investasi tetap memerlukan data utilisasi dan biaya.'],
 'digital':['Periksa relevansi layanan serta akses pengguna sebelum kampanye edukasi.','Uji kampanye adopsi digital berskala kecil dengan kelompok pembanding.','Evaluasi retensi atau penawaran layanan lanjutan setelah definisi metrik digital terkonfirmasi.'],
 'games':['Validasi pencatatan games, kemudian uji minat layanan gaming.','Uji paket gaming terbatas setelah data periode perubahan tervalidasi.','Periksa latensi, kualitas pengalaman, dan retensi; nilai tinggi belum membuktikan pendapatan tinggi.'],
 'video':['Validasi pencatatan video dan hambatan penggunaan.','Uji paket video dengan evaluasi perubahan penggunaan dan margin.','Tinjau kualitas streaming dan kapasitas jam sibuk setelah anomali akhir periode dijelaskan.']}
for c in cats:
    p = profiles[profiles.kategori==c]
    parts.append('<h2>'+c.capitalize()+' user: karakteristik setiap kelas</h2>')
    show = p[['kelas','site_bulan',c+'_user_median','target_p25','target_p75','subs_median','du_median','kontribusi_jumlah_teramati_pct']]
    parts.append(table(show))
    parts.append('<p>Median, P25, dan P75 menggunakan rata-rata harian site-bulan. Kontribusi adalah bagian dari jumlah metrik pada hari teramati; bukan pangsa pelanggan unik dan dipengaruhi cakupan hari.</p>')
    for i,(_,r) in enumerate(p.iterrows()):
        others = ', '.join(f'{o} {num(r[o+"_user_median"])}' for o in cats if o!=c)
        parts.append(f'<p><strong>{r["kelas"]}:</strong> median {c} {num(r[c+"_user_median"])}; 50% observasi di antara {num(r.target_p25)}–{num(r.target_p75)}. Median subs {num(r.subs_median)} dan du {num(r.du_median)}. Profil kategori lain: {others}. Median hari tercatat {num(r.median_hari_teramati)}. Hipotesis tindak lanjut: {actions[c][i]}</p>')
    high = p[p.kelas=='Tinggi'].iloc[0]
    low = p[p.kelas=='Rendah'].iloc[0]
    parts.append(f'<p>Kelas Tinggi menyumbang {num(high.kontribusi_jumlah_teramati_pct)}% jumlah {c} teramati. Rasio median Tinggi terhadap Rendah sebesar {num(high[c+"_user_median"]/low[c+"_user_median"])} kali. Pemisahan besaran ini diharapkan karena kelas dibentuk dari metrik yang sama; profil silang membantu melihat konteks ukuran site.</p>')

parts += ['<h2>Perubahan kelas dan kualitas data</h2><img src="distribusi_kelas.png"><img src="tren_harian.png">',
 '<p>Tren memakai semua site yang tersedia pada masing-masing tanggal, sehingga perubahan juga dapat berasal dari komposisi site. Juni dan Juli perlu kehati-hatian khusus pada games/video. Garis merah menandai 28 Juni 2024 sebagai awal periode perubahan berkelanjutan yang perlu ditinjau, bukan bukti penyebab perubahan. Grafik juga menunjukkan penurunan games/video mendekati nol pada beberapa tanggal sebelum 28 Juni, serta penurunan tajam digital dan payload pada tanggal lain. Karena itu, periode awal juga tidak dapat dianggap bebas masalah.</p>']
before = daily[(daily.event_date>='2024-06-21')&(daily.event_date<'2024-06-28')]
after = daily[(daily.event_date>='2024-06-28')&(daily.event_date<='2024-07-04')]
comparison=[]
for c in cats:
    a=before[c+'_user_site_avg'].mean(); b=after[c+'_user_site_avg'].mean()
    comparison.append({'kategori':c,'rerata_21_27_Juni':a,'rerata_28_Juni_4_Juli':b,'perubahan_pct':100*(b/a-1),
                       'tanggal_sebelum':int(before[c+'_user_site_avg'].count()),'tanggal_sesudah':int(after[c+'_user_site_avg'].count())})
parts += [table(pd.DataFrame(comparison)),
 '<p>Perbandingan di atas menggunakan rata-rata dari rata-rata harian lintas site (bobot sama per tanggal). Perubahan tidak otomatis berarti perubahan perilaku; perlu verifikasi definisi metrik, pipeline, dan site yang sama pada kedua periode.</p>',
 '<h2>Uji sensitivitas</h2><p>Ambang alternatif dihitung hanya dari site-bulan sebelum Juni 2024 untuk menghindari bulan yang bersinggungan dengan periode tinjauan. Persentase di bawah menunjukkan observasi seluruh periode yang berpindah kelas jika ambang tersebut digunakan. Ini pemeriksaan kepekaan, bukan bukti bahwa periode awal bebas masalah.</p>',table(sens),
 '<h2>Kesimpulan untuk kerja praktik</h2><p>Segmentasi menyediakan empat sudut pandang untuk memprioritaskan kajian site. Kelas Tinggi menunjukkan aktivitas historis relatif besar, kelas Sedang menjadi kandidat eksperimen peningkatan penggunaan, dan kelas Rendah memerlukan pemeriksaan basis serta hambatan penggunaan. Rekomendasi ini merupakan hipotesis yang perlu diuji, bukan hasil pengukuran dampak kampanye.</p>',
 '<p>Sebelum implementasi, konfirmasi kamus data dan satuan, penyebab tanggal hilang, serta perubahan games/video. Lengkapi dengan lokasi, kapasitas, kualitas jaringan, pendapatan dan biaya untuk keputusan bisnis. Jika dilanjutkan ke prediksi, gunakan pemisahan waktu dan tetapkan ambang pada data latih saja agar evaluasi tidak bocor.</p>',
 '<h2>Validasi</h2><p>CSV asli dibaca ulang bertahap. Pemeriksaan meliputi nilai kosong, nilai non-finite/negatif, keunikan hash kunci site-tanggal, rekonsiliasi jumlah metrik harian dan bulanan, serta pembacaan ulang hasil agregasi. Keunikan site-bulan, kelengkapan empat label, total anggota kelas, dan kontribusi 100% per kategori diverifikasi. Hash merupakan pemeriksaan praktis, bukan pemeriksaan pasangan kunci secara eksak. Tidak ada outlier dihapus atau tanggal hilang diisi nol.</p>']
html = '<!doctype html><html lang="id"><meta charset="utf-8"><title>Insight potensi site Telkomsel</title><style>body{font:16px/1.65 system-ui;color:#172b3a;max-width:1120px;margin:40px auto;padding:0 24px}h1{font-size:34px;color:#0f766e}h2{margin-top:40px;color:#115e59}table{border-collapse:collapse;width:100%;font-size:13px}th,td{padding:10px 8px;border-bottom:1px solid #cbd5e1;text-align:right}th{background:#edf5f4;overflow-wrap:anywhere}img{width:100%;margin:20px 0} @media print{body{font-size:11px}h2{break-after:avoid}tr,img{break-inside:avoid}}</style>'+''.join(parts)+'</html>'
(OUT/'laporan_insight.html').write_text(html)
print(thresholds.to_string(index=False))
print(profiles[['kategori','kelas','site_bulan','kontribusi_jumlah_teramati_pct']].to_string(index=False))
print(pd.DataFrame(comparison).to_string(index=False))
print('Validasi kelas dan kontribusi berhasil. Laporan: ', OUT/'laporan_insight.html')


## 4. Tabel ringkasan
Nilai satuan metrik harus dikonfirmasi melalui kamus data Telkomsel. File laporan HTML, tabel karakteristik, kelas setiap site-bulan, dan grafik tersedia di folder `hasil`.

In [ ]:
print(profiles.to_string(index=False))
print(sens.to_string(index=False))

## 5. Visual hasil

In [ ]:
from IPython.display import Image, display
display(Image(filename="hasil/distribusi_kelas.png"))
display(Image(filename="hasil/tren_harian.png"))

In [ ]:
from pathlib import Path

folder = Path('/content/drive/MyDrive/insight-kp/hasil')
for file in folder.glob('*.csv'):
    print(file.name)